In [ ]:
import gc
import torch
import importlib

import train
import dataset
import trainer

importlib.reload(train)
importlib.reload(dataset)
importlib.reload(trainer)

from train import parse_args, get_tokenizer, load_dataset, get_model
from dataset import QGDataset
from trainer import Trainer

def main(args=None):
    if args is None:
        args = parse_args()

    print(f"Using device: {args.device}")

    tokenizer = get_tokenizer(args.qg_model)
    dataset = load_dataset(args.dataset_path, dataset_type="squad", sample_ratio=0.0001)

    train_set = QGDataset(dataset["train"], args.max_length, args.pad_mask_id, tokenizer)
    valid_set = QGDataset(dataset["validation"], args.max_length, args.pad_mask_id, tokenizer)

    model = get_model(args.qg_model, args.device, tokenizer)

    trainer = Trainer(
        dataloader_workers=args.dataloader_workers,
        device=args.device,
        epochs=args.epochs,
        learning_rate=args.learning_rate,
        model=model,
        save_dir=args.save_dir,
        tokenizer=tokenizer,
        train_batch_size=args.train_batch_size,
        train_set=train_set,
        valid_batch_size=args.valid_batch_size,
        valid_set=valid_set
    )

    trainer.train()

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    trainer._save()
    print(f"Model telah disimpan di {args.save_dir}")

if __name__ == "__main__":
    main()